In [1]:
# -*- coding: utf-8 -*-
"""
Created on 2026-08-02
Revised on 2026-08-04

@author:       Oscar Trevizo
@institution:  Harvard Extension School — Graduate Data Science Program (2023)
@context:      Independent project — applying course concepts to real-world data
@environment:  Python 3.14.3 | myenv | MacBook Air M5

Python datetime Vignette
==========================

Description:
    Demonstrates Python's built-in datetime module: constructing a
    datetime, datetime.now(), dir() to show how many methods a datetime
    instance actually has, help() to look up what any one of those
    methods does, elapsed time via subtraction (timedelta), day-of-week
    and custom formatting via strftime(), and direct comparison
    operators. Then demonstrates datetime's one real limitation -- it
    requires a full year, month, and day, with no way to construct one
    from a year alone -- and builds a small PartialDateTime class to
    fill that gap. PartialDateTime doubles as a short OOP review: a
    constructor, instance attributes, the __str__ special method, and a
    has-a relationship, ahead of using the same class in an inheritance
    lesson elsewhere.

References:
    - GitHub: https://github.com/otrevizo/Python/tree/main/python_vignettes
    - https://docs.python.org/3/library/datetime.html

Revision History:
    2026-08-02  Original development
                - datetime basics, dir(datetime), timedelta, strftime,
                  comparisons
                - PartialDateTime class: __init__, __str__, as_datetime()
    2026-08-04  Added a help() example (help(datetime.weekday), then
                calling it) right after dir() and the elapsed-time demo
                -- dir() finds the method, help() explains it, then it's
                actually used, so readers are equipped to look up any
                other method from that dir() list themselves
"""


"\nCreated on 2026-08-02\nRevised on 2026-08-04\n\n@author:       Oscar Trevizo\n@institution:  Harvard Extension School — Graduate Data Science Program (2023)\n@context:      Independent project — applying course concepts to real-world data\n@environment:  Python 3.14.3 | myenv | MacBook Air M5\n\nPython datetime Vignette\n==========================\n\nDescription:\n    Demonstrates Python's built-in datetime module: constructing a\n    datetime, datetime.now(), dir() to show how many methods a datetime\n    instance actually has, help() to look up what any one of those\n    methods does, elapsed time via subtraction (timedelta), day-of-week\n    and custom formatting via strftime(), and direct comparison\n    operators. Then demonstrates datetime's one real limitation -- it\n    requires a full year, month, and day, with no way to construct one\n    from a year alone -- and builds a small PartialDateTime class to\n    fill that gap. PartialDateTime doubles as a short OOP review: a\n 

# Python `datetime` Vignette

Author: Oscar Trevizo

Date: August 2, 2026

## Reference

https://docs.python.org/3/library/datetime.html

Python's built-in `datetime` module is genuinely useful -- and it has
one real gap that motivates something worth building ourselves.

## The basics

`datetime` is a **class** -- `datetime(...)` *instantiates* an object
with year, month, day, hour, minute, second all bundled together in one
place.

In [2]:
from datetime import datetime

d = datetime(1968, 7, 31, 14, 30)   # the day the Beatles recorded "Hey Jude"
print(d)
print(type(d))

1968-07-31 14:30:00
<class 'datetime.datetime'>


In [3]:
# Since datetime is a class, it comes with a whole set of built-in
# methods -- dir() lists them all.
public_names = [name for name in dir(datetime) if not name.startswith("_")]
print(len(public_names), "public names available on datetime")
print(public_names)

40 public names available on datetime
['astimezone', 'combine', 'ctime', 'date', 'day', 'dst', 'fold', 'fromisocalendar', 'fromisoformat', 'fromordinal', 'fromtimestamp', 'hour', 'isocalendar', 'isoformat', 'isoweekday', 'max', 'microsecond', 'min', 'minute', 'month', 'now', 'replace', 'resolution', 'second', 'strftime', 'strptime', 'time', 'timestamp', 'timetuple', 'timetz', 'today', 'toordinal', 'tzinfo', 'tzname', 'utcfromtimestamp', 'utcnow', 'utcoffset', 'utctimetuple', 'weekday', 'year']


In [4]:
# datetime also knows what time it is right now
now = datetime.now()
print(now)

2026-08-04 13:25:32.845813


## How much time has passed?

Subtract one `datetime` from another and Python hands you back a
`timedelta` -- the actual elapsed time between them, correctly handling
leap years and everything else, with zero manual date math.

In [5]:
recorded = datetime(1968, 7, 31)   # the Beatles recorded "Hey Jude" on this date

elapsed = datetime.now() - recorded
print(f"Days elapsed: {elapsed.days:,}")

Days elapsed: 21,188


### Don't know what a method does? `help()` does.

`dir()` told us the *names* of everything available on `datetime` --
`help()` tells you what any one of them actually *does*: what
arguments it takes, what it returns, explained in plain text. This
works on literally any method from that `dir()` list above.

In [6]:
help(datetime.weekday)

Help on method descriptor weekday:

weekday(self, /) unbound datetime.date method
    Return the day of the week represented by the date.
    Monday == 0 ... Sunday == 6



In [7]:
# Now that we know what it does, use it -- Monday=0 ... Sunday=6
print(recorded.weekday())

2


## What day of the week was that?

`weekday()` above gave us a number -- `strftime()` ("string format
time") gets us the actual name, and can turn a `datetime` into whatever
text format we want.

In [8]:
print("That date fell on a", recorded.strftime("%A"))
print("Nicely formatted:", recorded.strftime("%B %d, %Y"))

That date fell on a Wednesday
Nicely formatted: July 31, 1968


## Are two dates in order?

`datetime` objects compare directly with `<`, `>`, `==` -- no need to
pull out year/month/day and compare those by hand.

In [10]:
released = datetime(1968, 8, 26)   # "Hey Jude" single release date

print("Earlier date comes first?", recorded < released)
print("Days between the two:", (released - recorded).days)

Earlier date comes first? True
Days between the two: 26


## The gap

All of that is genuinely nice -- *as long as you know the full date*.
But real historical dates are often only partly known. What if we only
know the **year** something happened?

In [11]:
try:
    d = datetime(1969)
except TypeError as e:
    print(f"TypeError: {e}")

TypeError: function missing required argument 'month' (pos 2)


`datetime` requires year, month, *and* day -- there's no "just give me
the year" mode. The only way around it is to make something up for the
missing pieces (`datetime(1969, 1, 1)`), but then that guess looks
*identical* to a genuinely known January 1st -- there's no way to tell
"really January 1st" from "we don't actually know." That's a real gap,
not something obvious we're missing.

## Filling the gap: a class of our own

Since `datetime` can't represent "partially known," we'll build
something that can. This is also a nice quick review of what a class
actually is: a way to bundle related data (`year`, `month`, `day`, ...)
together with the behavior that makes sense for that data (`__str__` to
display it, a method to convert it).

- The **constructor** (`__init__`) sets up the instance -- `year` is
  required, everything else defaults to `None`, meaning "unknown."
- `__str__` is a **special method** -- define it, and `print()` and
  f-strings automatically know how to display the object, using only
  the precision that's actually known.
- `as_datetime()` is an ordinary method that converts to a *real*
  `datetime.datetime` when one is genuinely needed, filling in any
  unknown month/day/hour/minute with 1/1/0/0.

In [12]:
class PartialDateTime:
    """A date/time that might only be partially known."""
    def __init__(self, year, month=None, day=None, hour=None, minute=None):
        self.year = year
        self.month = month     # None if unknown
        self.day = day          # None if unknown
        self.hour = hour         # None if unknown
        self.minute = minute      # None if unknown

    def __str__(self):
        text = f"{self.year}"
        for value in (self.month, self.day):
            if value is None:
                break
            text += f"-{value:02d}"
        if self.hour is not None:
            text += f" {self.hour:02d}"
            if self.minute is not None:
                text += f":{self.minute:02d}"
        return text

    def as_datetime(self):
        """Convert to a real datetime.datetime. Fills any unknown
        month/day/hour/minute with 1/1/0/0 -- once converted, "really
        January 1st" and "unknown, defaulted to January 1st" look
        identical, so prefer str(self) when you just need to display
        the date."""
        return datetime(self.year, self.month or 1, self.day or 1,
                         self.hour or 0, self.minute or 0)

In [13]:
# Voila -- now we CAN represent "we only know the year"
year_only = PartialDateTime(1969)                    # e.g. "Come Together" -- only the year is known
year_month = PartialDateTime(1968, 7)                  # e.g. "Hey Jude," recorded sometime in July 1968
full_precision = PartialDateTime(1968, 7, 31, 14, 30)   # "Hey Jude," recorded on this exact date/time

for sample in (year_only, year_month, full_precision):
    print(f"{sample}  -->  as_datetime(): {sample.as_datetime()}")

1969  -->  as_datetime(): 1969-01-01 00:00:00
1968-07  -->  as_datetime(): 1968-07-01 00:00:00
1968-07-31 14:30  -->  as_datetime(): 1968-07-31 14:30:00


## Takeaways

- `datetime` is a class -- instantiate it, and you get an object with
  dozens of built-in methods for formatting, comparing, and doing real
  date math for free.
- Its one real limitation: no partial precision. A `PartialDateTime`
  class (has-a year/month/day/hour/minute, mostly optional) fills that
  gap, and along the way reviews the core pieces of a class: a
  constructor, instance attributes, and the `__str__` special method.

I use this `PartialDateTime` class in a follow-up lesson on
inheritance, where a few different classes each has-a partially-known
date.